# Reasoning-трейсы (Think2SQL) — 7B

Тест гипотезы Think2SQL: помогает ли явное рассуждение маленькой SFT-модели.
Таргет = `<рассуждение>\nSQL:\n<gold sql>`; на оценке SQL берётся ПОСЛЕ маркера `SQL:`.
Сравнивается с обычным 7B (38.09% ± 1.37).

**Данные:** залей датасет, где лежат `train_reasoning.json` (от generate_reasoning.py),
`val.json`, `bird_large.json`, профили. Поправь `DATA_DIR`. GPU On + Internet On.

In [ ]:
!pip install -q -U transformers peft bitsandbytes accelerate trl datasets

In [ ]:
import json
import re
from dataclasses import fields
from pathlib import Path

import torch

MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"
DATA_DIR = Path("/kaggle/input/datasets/vorange/db2model-lora")  # ПОПРАВЬ
OUT_DIR = Path("/kaggle/working")
ADAPTER_DIR = OUT_DIR / "adapter"

train_pairs = json.loads((DATA_DIR / "train_reasoning.json").read_text(encoding="utf-8"))
val_pairs = json.loads((DATA_DIR / "val.json").read_text(encoding="utf-8"))
bird = json.loads((DATA_DIR / "bird_large.json").read_text(encoding="utf-8"))
DBS = ["financial", "toxicology", "codebase_community"]
profiles = {db: json.loads((DATA_DIR / f"{db}_profile.json").read_text(encoding="utf-8")) for db in DBS}
print("GPU:", torch.cuda.get_device_name(0), "| reasoning-пар:", len(train_pairs))

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=COMPUTE_DTYPE, bnb_4bit_use_double_quant=True
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
kw = dict(quantization_config=bnb, device_map={"": 0})
try:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=COMPUTE_DTYPE, **kw)
except TypeError:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=COMPUTE_DTYPE, **kw)
model.config.use_cache = False
print("память на GPU:", round(torch.cuda.memory_allocated() / 1e9, 2), "ГБ")

In [ ]:
# Промпт reasoning-арма: без схемы (знание в весах), просим сначала рассуждение,
# потом финальный запрос после строки "SQL:".
def build_prompt(db, question):
    system = (
        f"You are a PostgreSQL expert for the database `{db}`. "
        'Think step by step, then give the final query after a line "SQL:".'
    )
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": system}, {"role": "user", "content": question}],
        tokenize=False,
        add_generation_prompt=True,
    )


def generate(gen_model, prompt, max_new_tokens=400):
    inputs = tokenizer(prompt, return_tensors="pt").to(gen_model.device)
    with torch.no_grad():
        out = gen_model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id
        )
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True)
    return text, inputs["input_ids"].shape[1]


FENCED = re.compile(r"```(?:sql)?\s*(.*?)```", re.DOTALL | re.IGNORECASE)
STATEMENT = re.compile(r"\b(WITH|SELECT)\b", re.IGNORECASE)


def extract_sql(text):
    # берём часть ПОСЛЕ последнего маркера "SQL:", иначе SELECT из рассуждения
    if "SQL:" in text:
        text = text.rsplit("SQL:", 1)[1]
    text = text.strip()
    m = FENCED.search(text)
    if m:
        text = m.group(1).strip()
    s = STATEMENT.search(text)
    if s:
        text = text[s.start() :]
    return text.strip().rstrip(";").strip()

In [ ]:
from datasets import Dataset


# Таргет: рассуждение + маркер + gold SQL. Схему в промпт не кладём.
def to_text(p):
    prompt = build_prompt(p["db_id"], p["question"])
    target = p["reasoning"].strip() + "\nSQL:\n" + p["sql"]
    return {"text": prompt + target + tokenizer.eos_token}


train_ds = Dataset.from_list([to_text(p) for p in train_pairs])
print(train_ds)
print("\nпример:\n", train_ds[0]["text"][:500])

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

SUPPORTED = {f.name for f in fields(SFTConfig)}
model = prepare_model_for_kbit_training(model)
use_bf16 = COMPUTE_DTYPE is torch.bfloat16
kwargs = dict(
    output_dir=str(OUT_DIR / "ckpt"),
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    lr_scheduler_type="cosine",
    warmup_steps=10,
    logging_steps=50,
    save_strategy="no",
    bf16=use_bf16,
    fp16=not use_bf16,
    optim="paged_adamw_8bit",
    dataset_text_field="text",
    report_to="none",
    max_length=640,
    seed=0,
)
args = SFTConfig(**{k: v for k, v in kwargs.items() if k in SUPPORTED})
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    peft_config=LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    ),
)
trainer.train()

In [ ]:
trainer.model.config.use_cache = True
trainer.model.eval()
questions = [q for q in bird if q["db_id"] in DBS]
print("вопросов:", len(questions))

preds = {}
for i, q in enumerate(questions, 1):
    question = f"question: {q['question']}, evidence (may be empty): {q['evidence']}"
    prompt = build_prompt(q["db_id"], question)
    text, _ = generate(trainer.model, prompt)
    preds[str(q["question_id"])] = extract_sql(text)
    if i % 20 == 0:
        print(f"  {i}/{len(questions)}")
out = OUT_DIR / "query_results_reason_lora.json"
out.write_text(json.dumps(preds, ensure_ascii=False, indent=2), encoding="utf-8")
print("готово ->", out, "| забери и посчитай EX дома")